# 패키지 import

In [1]:
### 필요 패키지 import ###
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
import pandas as pd
import PIL
import matplotlib.pyplot as plt
import os

In [2]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Data Load

In [3]:
seed = 7

classes_names = ["포트홀 없음", "포트홀", "보수 완료된 포트홀"]
train_data = pd.read_csv("Train_datasets_labels.csv")
valid_data = pd.read_csv("Validation_datasets_labels.csv")


train_data = train_data.sample(frac=1, random_state=seed).reset_index(drop=True)
valid_data = valid_data.sample(frac=1, random_state=seed).reset_index(drop=True)

print(len(train_data))
print(len(valid_data))

23389
300


# Define Data-Sets

In [4]:
class MyDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.x=0; self.y=0
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        file_path = self.df.iloc[idx][0]
        img = np.array(PIL.Image.open(train_data.iloc[idx][0])) / 255.
        label = torch.tensor(train_data.iloc[idx][1])
        
        self.x = torch.tensor(img).float()
        self.y = F.one_hot(label, num_classes=3).float()
        
        return self.x, self.y
    
train = MyDataset(train_data)
train_loader = DataLoader(train, batch_size = 16)
valid = MyDataset(valid_data)
valid_loader = DataLoader(valid, batch_size = 16)

In [13]:
for batch_idx, (img, label) in enumerate(valid_loader):
    print(batch_idx)
    print("img shape :", img.shape)
    print("label :", label.shape)

0
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
1
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
2
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
3
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
4
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
5
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
6
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
7
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
8
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
9
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
10
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
11
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
12
img shape : torch.Size([16, 320, 320, 3])
label : torch.Size([16, 3])
13
img shape : torch.Size([16, 320, 320, 3])
label : torch.Si

# Define Train,Valid Function

In [5]:
Loss_FN = nn.CrossEntropyLoss()

def train(EPOCHS, model, train_loader, opt):
    for epoch in range(1, EPOCHS+1):
        model.train()
        print("### EPOCH {} ###".format(epoch))
        for batch_idx, (img,label) in enumerate(train_loader):
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            output = model(img)
            loss = Loss_FN(output, label)
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            if batch_idx % 100 == 0:
                print("Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                    epoch, 
                    batch_idx * len(img), 
                    len(train_loader.dataset), 
                    100. * batch_idx / len(train_loader), 
                    loss.item()))
        t_loss, t_acc = evaluate(model, valid_loader)
        print("[{}] Test Loss: {:.4f}, accuracy: {:.2f}%\n".format(epoch, t_loss, t_acc))
                
def evaluate(model, valid_loader):
    model.eval()
    t_loss = 0
    correct = 0
    
    with torch.no_grad():
        for img, label in valid_loader:
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            output = model(img)
            t_loss += Loss_FN(output, label).item()
            
#             print(output.shape)
#             print(label.shape)
            correct += output.eq(label.view_as(output)).sum().item()
            
    t_loss /= len(valid_loader.dataset)
    t_acc = 100. * correct / len(valid_loader.dataset)
    return t_loss, t_acc

# Define Model

In [6]:
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

In [7]:
class Model1(nn.Module):
    def __init__(self):
        super(Model1, self).__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 8, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(in_channels = 8, out_channels = 16, kernel_size = 3, padding = 1)
        self.conv3 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1)
        self.conv4 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.fc1 = nn.Linear(20 * 20 * 64, 1024)
        self.fc2 = nn.Linear(1024, 64)
        self.fc3 = nn.Linear(64, 3)
        
        self.conv1_bn = nn.BatchNorm2d(8)
        self.conv2_bn = nn.BatchNorm2d(16)
        self.conv3_bn = nn.BatchNorm2d(32)
        self.conv4_bn = nn.BatchNorm2d(64)
        
        self.dropout_p = 0.2            # drop out 비율 값
        
    def forward(self, x):
        x = x.permute(0,3,1,2)
        x = self.conv1(x) # 320 * 320 * 3 -> 320 * 320 * 8
        x = self.conv1_bn(x)
        x = F.tanh(x)     # 320 * 320 * 8
        x = self.pool(x)  # 160 * 160 * 8
        
        x = self.conv2(x) # 160 * 160 * 8 -> 160 * 160 * 16
        x = self.conv2_bn(x)
        x = F.tanh(x)     # 160 * 160 * 16
        x = self.pool(x)  # 80  *  80 * 16
        
        x = self.conv3(x) # 80 * 80 * 16 -> 80 * 80 * 32
        x = self.conv3_bn(x)
        x = F.tanh(x)     # 80 * 80 * 32
        x = self.pool(x)  # 40 * 40 * 32
        
        x = self.conv4(x) # 40 * 40 * 32 -> 40* 40 * 64
        x = self.conv4_bn(x)
        x = F.tanh(x)     # 40 * 40 * 64
        x = self.pool(x)  # 20 * 20 * 64
        
        x = x.contiguous().view(-1, 20*20*64)
        x = self.fc1(x)
        x = F.dropout(x, p = self.dropout_p) # Drop out 적용
        x = F.relu(x)
        x = self.fc2(x)
        x = F.dropout(x, p = self.dropout_p) # Drop out 적용
        x = F.relu(x)
        x = self.fc3(x)
        x = F.log_softmax(x)
        return x

In [8]:
model = Model1().to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr = 0.001)
print("DEVICE: ", DEVICE)
print("MODEL: ", model)

DEVICE:  cuda
MODEL:  Model1(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=25600, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
  (conv1_bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2_bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)


In [9]:
train(15, model, train_loader, optimizer)

### EPOCH 1 ###


C:\anaconda3\envs\JSY_GPU\lib\site-packages\torch\nn\functional.py:1795: UserWarning: nn.functional.tanh is deprecated. Use torch.tanh instead.
  warnings.warn("nn.functional.tanh is deprecated. Use torch.tanh instead.")
C:\anaconda3\envs\JSY_GPU\lib\site-packages\ipykernel_launcher.py:50: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.


Train Epoch: 1 [0/23389 (0%)]	Loss: 1.088867
Train Epoch: 1 [1600/23389 (7%)]	Loss: 0.863496
Train Epoch: 1 [3200/23389 (14%)]	Loss: 1.121817
Train Epoch: 1 [4800/23389 (21%)]	Loss: 0.561395
Train Epoch: 1 [6400/23389 (27%)]	Loss: 0.820558
Train Epoch: 1 [8000/23389 (34%)]	Loss: 0.643949
Train Epoch: 1 [9600/23389 (41%)]	Loss: 0.635803
Train Epoch: 1 [11200/23389 (48%)]	Loss: 0.574713
Train Epoch: 1 [12800/23389 (55%)]	Loss: 1.061307
Train Epoch: 1 [14400/23389 (62%)]	Loss: 0.539236
Train Epoch: 1 [16000/23389 (68%)]	Loss: 0.649096
Train Epoch: 1 [17600/23389 (75%)]	Loss: 0.787117
Train Epoch: 1 [19200/23389 (82%)]	Loss: 0.564315
Train Epoch: 1 [20800/23389 (89%)]	Loss: 0.417815
Train Epoch: 1 [22400/23389 (96%)]	Loss: 0.927338
[1] Test Loss: 0.0541, accuracy: 0.00%

### EPOCH 2 ###
Train Epoch: 2 [0/23389 (0%)]	Loss: 0.552337
Train Epoch: 2 [1600/23389 (7%)]	Loss: 0.683965
Train Epoch: 2 [3200/23389 (14%)]	Loss: 0.886587
Train Epoch: 2 [4800/23389 (21%)]	Loss: 0.321077
Train Epoch: 2 

Train Epoch: 11 [8000/23389 (34%)]	Loss: 0.516356
Train Epoch: 11 [9600/23389 (41%)]	Loss: 0.398976
Train Epoch: 11 [11200/23389 (48%)]	Loss: 0.369750
Train Epoch: 11 [12800/23389 (55%)]	Loss: 0.388581
Train Epoch: 11 [14400/23389 (62%)]	Loss: 0.456319
Train Epoch: 11 [16000/23389 (68%)]	Loss: 0.495987
Train Epoch: 11 [17600/23389 (75%)]	Loss: 0.373299
Train Epoch: 11 [19200/23389 (82%)]	Loss: 0.209426
Train Epoch: 11 [20800/23389 (89%)]	Loss: 0.239180
Train Epoch: 11 [22400/23389 (96%)]	Loss: 0.703186
[11] Test Loss: 0.0243, accuracy: 0.00%

### EPOCH 12 ###
Train Epoch: 12 [0/23389 (0%)]	Loss: 0.370342
Train Epoch: 12 [1600/23389 (7%)]	Loss: 0.377183
Train Epoch: 12 [3200/23389 (14%)]	Loss: 0.705361
Train Epoch: 12 [4800/23389 (21%)]	Loss: 0.120561
Train Epoch: 12 [6400/23389 (27%)]	Loss: 0.099973
Train Epoch: 12 [8000/23389 (34%)]	Loss: 0.464462
Train Epoch: 12 [9600/23389 (41%)]	Loss: 0.258686
Train Epoch: 12 [11200/23389 (48%)]	Loss: 0.216063
Train Epoch: 12 [12800/23389 (55%)]	Lo